# Data setup notebook 

In [ ]:
from pathlib import Path
import os   
import subprocess 
import sys

def local_run():
    if os.path.exists("/content"):
        # running in Google Colab
        return False
    else:
        # running locally
        return True

def get_base_dir():
    if local_run():
        # running locally
        return Path(__vsc_ipynb_file__).resolve().parent.parent
    else:
        # running in Google Colab
        return Path("/content/deforestation-unet")

if not local_run() :
    import getpass
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
if not local_run():
    token = getpass.getpass("Enter your GitHub token: ")
    base_dir = get_base_dir()
    repo_url = f"https://{token}@github.com/barbara-barta/deforestation-unet.git"
    if not base_dir.exists():
        print("Repository does not exist, cloning it now.")
        subprocess.run(["git", "clone", repo_url], check=True)
    else:
        print("Repository already exists, pulling the latest changes.")
        subprocess.run(["git", "-C", str(base_dir), "pull"], check=True)

In [ ]:
if not local_run():
    %pip install -r /content/deforestation-unet/requirements.txt

If this is your first run, please restart the kernel after all the packages are downloaded. 

In [ ]:
paths = {
    "rgb": base_dir / "data" / "rgb" /"raw",
    "am4": base_dir / "data" / "am4" / "raw",
    "at4": base_dir / "data" / "at4" / "raw",    
}

downloads = {
    "RGB": ["https://doi.org/10.5281/zenodo.3233081", "Amazon Forest Dataset.rar"],
    "AM4": ["https://doi.org/10.5281/zenodo.4498086","AMAZON.rar"],
    "AT4": ["https://doi.org/10.5281/zenodo.4498086","ATLANTIC FOREST.rar"]}

for path in paths.values():
    path.mkdir(parents=True, exist_ok=True)

for dataset, [url, filename] in downloads.items():      
    data_present = list(Path(paths[dataset]).glob(filename))
    if not data_present:
        print(f"Downloading {dataset} data...")
        cmd = ["zenodo_get", "-o", str(paths[dataset]), "-g", filename] + [url]
        subprocess.run(cmd, check=True)
        print(f"Dataset {dataset} downloaded to {str(paths[dataset])}.")
    else:
        print(
            f"{dataset} data already exists at {str(paths[dataset])}, skipping download."
        )
 
    unziped_filename = filename.rsplit(".", 1)[0]
    unziped_data_present = list(Path(paths[dataset]).glob(unziped_filename))
    
    if not unziped_data_present:
        print(f"Unzipping {dataset} data...")
        cmd = ["unrar", "x", "-o+", str(paths[dataset] / filename), str(paths[dataset])]
        subprocess.run(cmd, check=True)
        print(f"Dataset {dataset} unzipped in {str(paths[dataset])}.")